# Worker Preflight Check

This notebook performs comprehensive preflight checks to verify the worker environment is properly configured and ready for video encoding tasks.

In [21]:
from pathlib import Path
import logging
import os
import subprocess

In [22]:
response = (True, {
    'before_file_size': 757,
    'directory_path': '/boilmedia/Media 1',
    'file_guid': '05d39d91-1582-49c2-a4e9-8350e70488f1',
    'input_file_name': 'test_file_01.mp4',
    'output_file_name': 'test.mkv'
})

# Or unpack it into separate variables
success, data = response

# Access specific values
before_file_size = data['before_file_size']
directory_path = data['directory_path']
file_guid = data['file_guid']
input_file_name = data['input_file_name']
output_file_name = data['output_file_name']

logging.debug(before_file_size)
logging.debug(directory_path)
logging.debug(file_guid)
logging.debug(input_file_name)
logging.debug(output_file_name)


In [23]:
def get_file_size_kb(directory_path, input_file_name):
    try:
        file_path = os.path.join(directory_path, input_file_name)
        file_size_bytes = Path(file_path).stat().st_size
        file_size_kb = int(file_size_bytes / 1024)
        return file_size_kb
    except FileNotFoundError:
        logging.debug(f"✗ File not found: {file_path}")
        return 0
    except Exception as e:
        logging.debug(f"✗ Error getting file size: {e}")
        return 0
    
    
def validate_hash(directory_path, input_file_name, before_file_size):
    current_size_kb = get_file_size_kb(directory_path, input_file_name)   
    if current_size_kb == before_file_size:
        logging.debug(f"✓ Preflight check passed: {current_size_kb} KB == {before_file_size} KB")
        return True
    else:
        logging.debug(f"✗ Preflight check failed: {current_size_kb} KB != {before_file_size} KB")
        return False

logging.debug(validate_hash(directory_path, input_file_name, before_file_size))


In [24]:
def validate_video(directory_path, input_file_name):
    try:
        file_path = os.path.join(directory_path, input_file_name)
        command = 'ffmpeg -v error -i "' + file_path + '" -f null -'
        result = subprocess.run(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if result.stdout or result.stderr:
            logging.debug('File failed video integrity check')
            return False
        else:
            logging.debug('File passed video integrity check')
            return True
    except Exception as e:
        logging.debug(f"Error during video integrity check: {e}")
        return False
    
logging.debug(validate_video(directory_path, input_file_name))


Actual Preflight Check

In [25]:
def validate_preflight(directory_path, input_file_name, before_file_size):
    video_ok = validate_video(directory_path, input_file_name)
    hash_ok = validate_hash(directory_path, input_file_name, before_file_size)

    if video_ok and hash_ok:
        logging.debug("✓ Preflight validation passed (video + hash)")
        return True

    logging.debug("✗ Preflight validation failed (video + hash)")
    return False

logging.debug(validate_preflight(directory_path, input_file_name, before_file_size))
